# 第13章 代码教学：代码生成（Code Generation）

> 目标：
> 1) 统一 Prompt 模板
> 2) 生成代码并自动测试
> 3) 失败反馈 -> 迭代修复（Self-Refine）
> 4) 小样本 pass@k 估计


## 0. 环境准备


In [1]:
# !pip install -U transformers accelerate
import ast, builtins, math, re, traceback
import numpy as np, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
np.random.seed(42); torch.manual_seed(42)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device=', device)


C:\Users\250010108\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


device= cpu


## 1. 自动选择可用模型（带回退）

优先选择小型代码模型，失败则回退到通用模型。


In [2]:
CANDIDATES = ['bigcode/tiny_starcoder_py','Salesforce/codegen-350M-mono','gpt2']

tok = model = None
ACTIVE = None
for name in CANDIDATES:
    try:
        print('trying', name)
        tok = AutoTokenizer.from_pretrained(name)
        if tok.pad_token is None: tok.pad_token = tok.eos_token
        model = AutoModelForCausalLM.from_pretrained(name).to(device)
        model.eval(); ACTIVE = name; break
    except Exception as e:
        print('failed', name, str(e)[:80])

if ACTIVE is None:
    raise RuntimeError('no model loaded')
print('selected=', ACTIVE)


trying bigcode/tiny_starcoder_py


Loading weights:   0%|          | 0/244 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/244 [00:00<00:00, 23967.45it/s, Materializing param=transformer.h.0.attn.c_attn.bias]

Loading weights:   0%|          | 1/244 [00:00<00:00, 1367.56it/s, Materializing param=transformer.h.0.attn.c_attn.bias] 

Loading weights:   1%|          | 2/244 [00:00<00:00, 1681.76it/s, Materializing param=transformer.h.0.attn.c_attn.weight]

...

Loading weights: 100%|█████████▉| 243/244 [00:00<00:00, 1966.70it/s, Materializing param=transformer.wpe.weight]

Loading weights: 100%|██████████| 244/244 [00:00<00:00, 1966.70it/s, Materializing param=transformer.wte.weight]

Loading weights: 100%|██████████| 244/244 [00:00<00:00, 1966.70it/s, Materializing param=transformer.wte.weight]

Loading weights: 100%|██████████| 244/244 [00:00<00:00, 2020.05it/s, Materializing param=transformer.wte.weight]

selected= bigcode/tiny_starcoder_py


## 2. 任务与测试集

每个任务自带测试用例，便于自动验收。


In [3]:
tasks = [
    {"name":"fib","inst":"Write fib(n) with fib(0)=0,fib(1)=1.","tests":"assert fib(0)==0\nassert fib(1)==1\nassert fib(7)==13\nassert fib(10)==55"},
    {"name":"is_prime","inst":"Write is_prime(n). Return True only for primes.","tests":"assert is_prime(2) is True\nassert is_prime(3) is True\nassert is_prime(4) is False\nassert is_prime(29) is True\nassert is_prime(1) is False"},
    {"name":"word_count","inst":"Write word_count(text) returning lowercase word frequency dict split by punctuation/spaces.","tests":"r=word_count('AI ai, systems! systems')\nassert r['ai']==2\nassert r['systems']==2"},
]
print('tasks=', [t['name'] for t in tasks])


tasks= ['fib', 'is_prime', 'word_count']


## 3. Prompt 模板与生成函数


In [4]:
def build_prompt(inst, feedback=''):
    # 约束：只输出代码，不要解释文本
    p = "You are a precise Python coder. Output ONLY Python code.\n" + f"Task: {inst}\n"
    if feedback:
        p += f"Previous error:\n{feedback}\nFix it.\n"
    return p

@torch.no_grad()
def generate(prompt, max_new_tokens=220, temperature=0.0, top_p=0.95):
    x = tok(prompt, return_tensors='pt').to(device)
    y = model.generate(
        **x,
        max_new_tokens=max_new_tokens,
        do_sample=temperature>0,
        temperature=max(temperature,1e-5),
        top_p=top_p,
        pad_token_id=tok.eos_token_id,
    )
    gen_ids = y[0][x['input_ids'].shape[1]:]
    return tok.decode(gen_ids, skip_special_tokens=True)


## 4. 代码提取 + 语法检查


In [5]:
def extract_code(text):
    # 兼容 markdown 代码块输出
    b = re.findall(r"```python(.*?)```", text, flags=re.S|re.I)
    if b: return b[0].strip()
    b2 = re.findall(r"```(.*?)```", text, flags=re.S)
    if b2: return b2[0].strip()
    return text.strip()


def syntax_ok(code):
    try:
        ast.parse(code); return True, 'OK'
    except Exception as e:
        return False, str(e)


## 5. 自动测试执行器

教学版使用受限 builtins；生产建议使用更严格沙箱。


In [6]:
SAFE = {k:getattr(builtins,k) for k in ['abs','all','any','bool','dict','enumerate','float','int','len','list','max','min','pow','print','range','set','str','sum','tuple','zip']}
SAFE['__import__'] = builtins.__import__

def run_tests(code_text, tests_text):
    ok, msg = syntax_ok(code_text)
    if not ok: return False, 'SyntaxError: ' + msg
    g = {'__builtins__': SAFE, 'math': math, 're': re}; l = {}
    try:
        exec(code_text, g, l)
        exec(tests_text, g, l)
        return True, 'PASS'
    except Exception:
        return False, traceback.format_exc(limit=1)


## 6. 基线：一次生成（One-shot）


In [7]:
baseline = []
for t in tasks:
    raw = generate(build_prompt(t['inst']), temperature=0.0)
    code_text = extract_code(raw)
    ok, rep = run_tests(code_text, t['tests'])
    baseline.append({'task': t['name'], 'ok': ok, 'report': rep[:120], 'code': code_text})

for r in baseline:
    print(r['task'], '->', r['ok'], '|', r['report'])


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


fib -> False | SyntaxError: unterminated triple-quoted string literal (detected at line 13) (<unknown>, line 3)
is_prime -> False | SyntaxError: invalid syntax (<unknown>, line 1)
word_count -> False | SyntaxError: invalid syntax (<unknown>, line 7)


## 7. 失败反馈驱动的迭代修复（Self-Refine）


In [8]:
def solve_refine(task, max_round=3):
    fb = ''
    hist = []
    for rd in range(1, max_round+1):
        raw = generate(build_prompt(task['inst'], fb), temperature=0.2 if rd>1 else 0.0)
        code_text = extract_code(raw)
        ok, rep = run_tests(code_text, task['tests'])
        hist.append({'round': rd, 'ok': ok, 'report': rep, 'code': code_text})
        if ok: return True, hist
        fb = rep
    return False, hist

refined = []
for t in tasks:
    ok, h = solve_refine(t, 3)
    refined.append({'task': t['name'], 'ok': ok, 'rounds': len(h)})
print(refined)


[{'task': 'fib', 'ok': False, 'rounds': 3}, {'task': 'is_prime', 'ok': False, 'rounds': 3}, {'task': 'word_count', 'ok': False, 'rounds': 3}]


## 8. pass@k 估计（小样本）


In [9]:
def pass_at_k(task, k=5, temp=0.8):
    for _ in range(k):
        raw = generate(build_prompt(task['inst']), temperature=temp, top_p=0.95)
        ok, _ = run_tests(extract_code(raw), task['tests'])
        if ok: return 1
    return 0

K = 5
print({t['name']: pass_at_k(t, K, 0.8) for t in tasks})


{'fib': 0, 'is_prime': 0, 'word_count': 0}


## 9. 示例代码审阅


In [10]:
for r in baseline:
    if r['ok']:
        print('task=', r['task'])
        print(r['code'])
        break


## 10. 练习
1) 增加边界测试；2) 约束复杂度；3) 对比不同模型的 pass@k。
